# Train DSSM

In [11]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

In [12]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.context_aware_recommender import DSSM
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [13]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "target" 
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

## Create dataset

In [17]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "LABEL_FIELD": "label",
    "threshold": {"rating": 1.0},
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id", "rating"],
        "user": ["user_id", "category"],
        "item": ["item_id", "store", "price"],
    },
    "numerical_features": ["price"],
    "train_neg_sample_args": {
        "distribution": "uniform",
        "sample_num": 1,
    },
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 1024 * 32,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": "full",
    },
    "metrics": ["Recall", "NDCG", "MRR"],
    "valid_metric": "NDCG@10",
    "seed": SEED,
}

config: Config = Config(model="DSSM", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [18]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work becaus

## Train DSSM

In [19]:
model: DSSM = DSSM(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
18 Jun 19:59    INFO  epoch 0 training [time: 3.60s, train loss: 108.4182]
18 Jun 20:00    INFO  epoch 0 evaluating [time: 63.44s, valid_score: 0.000700]
18 Jun 20:00    INFO  valid result: 
recall@10 : 0.0014    ndcg@10 : 0.0007    mrr@10 : 0.0007
18 Jun 20:00    INFO  Saving current: saved/DSSM-Jun-18-2026_19-59-17.pth
18 Jun 20:00    INFO  epoch 1 training [time: 3.30s, train loss: 105.2373]
18 Jun 20:01    INFO  epoch 1 evaluating [time: 63.19s, valid_score: 0.001900]
18 Jun 20:01    INFO  valid result: 
recall@10 : 0.003    ndcg@10 : 0.0019    mrr@10 : 0.0021
18 Jun 20:01    INFO  Saving current: saved/DSSM-Jun-18-2026_19-59-17.pth
18 Jun 20:01    INFO  epoch 2 training [time: 3.41s, train loss

In [20]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0070
Best valid result:
  recall@10: 0.0108
  ndcg@10: 0.0070
  mrr@10: 0.0069


## Evaluate on test set

In [21]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

Test results (Overall):
  recall@10: 0.0101
  ndcg@10: 0.0081
  mrr@10: 0.0101


In [22]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    '''
    Evaluate the model on a subset of interactions defined by `mask`.
    '''
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold", tok["2"]: "new"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)

18 Jun 20:27    INFO  Loading model structure and parameters from saved/DSSM-Jun-18-2026_19-59-17.pth



Evaluation (warm)
--------------------
  Interactions: 4495
  recall@10: 0.0068
  ndcg@10: 0.0045
  mrr@10: 0.0052


18 Jun 20:27    INFO  Loading model structure and parameters from saved/DSSM-Jun-18-2026_19-59-17.pth



Evaluation (cold)
--------------------
  Interactions: 2678
  recall@10: 0.0062
  ndcg@10: 0.0047
  mrr@10: 0.0055


18 Jun 20:28    INFO  Loading model structure and parameters from saved/DSSM-Jun-18-2026_19-59-17.pth



Evaluation (new)
--------------------
  Interactions: 2826
  recall@10: 0.0172
  ndcg@10: 0.0149
  mrr@10: 0.0193
